# 03: Publication tables (v2)

Builds tables from existing files in `outputs/py/` only. Nothing is re-estimated.
Coefficients come from `estimates.csv`. Descriptives are computed from
`dk_clean_v2.parquet`. D&K values come only from `data/comparisons/dk_published_penalty.csv`.

Output per table, in `outputs/tables/`, with `_v2` suffixes so earlier tables are not overwritten:
HTML (Word), `.tex` (Overleaf, via `as_latex()`), and the underlying numbers as CSV.
PNG export is not in this notebook yet. It needs Selenium, which is not installed (Chrome is present).

In [ ]:
import os
import math
import numpy as np
import pandas as pd
from great_tables import GT

OUT = "../outputs"
os.makedirs(f"{OUT}/tables", exist_ok=True)

estimates = pd.read_csv(f"{OUT}/py/estimates.csv")
bench = pd.read_csv("../data/comparisons/dk_published_penalty.csv")

KQ_COV = ["log_wage", "outsourced", "union", "parttime", "educ_cat", "female", "black", "hispanic",
          "age", "age2", "msa", "central_city", "cc_unknown", "geo_unidentified", "gov_employee"]
COLS = ["occupation", "YEAR", "MONTH", "outsourced", "EARNWT", "ym", "real_wage_trimmed",
        "union", "parttime", "age", "female", "black", "hispanic", "educ_cat"] + KQ_COV
COLS = list(dict.fromkeys(COLS))
dk_base = pd.read_parquet(f"{OUT}/py/dk_clean_v2.parquet", columns=COLS)
print("base sample:", dk_base.shape)

In [ ]:
def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

def coef_cell(est, se, p):
    return f"{est:.3f}{stars(p)} ({se:.3f})"

def p_two_sided(z):
    return math.erfc(abs(z) / math.sqrt(2))

def get_row(df, **kw):
    m = np.ones(len(df), dtype=bool)
    for k, v in kw.items():
        m &= (df[k] == v).to_numpy()
    sub = df[m]
    assert len(sub) == 1, (kw, len(sub))
    return sub.iloc[0]

def style(gt_obj):
    return (gt_obj
            .opt_table_font(font=["Times New Roman", "Times", "serif"])
            .tab_options(table_font_size="10pt"))

def save_outputs(gt_obj, raw, stem):
    with open(f"{OUT}/tables/{stem}.html", "w") as f:
        f.write(gt_obj.as_raw_html())
    with open(f"{OUT}/tables/{stem}.tex", "w") as f:
        f.write(gt_obj.as_latex())
    raw.to_csv(f"{OUT}/tables/{stem}.csv", index=False)
    print("saved", stem)

SOURCE_FE = ("FE: state x year and clustering by survey month (ym) apply to regression tables only; "
             "Tables 1-2 are descriptive. Stars: * p<.10, ** p<.05, *** p<.01, two-tailed.")

In [ ]:
periods5 = ["2001-2005", "2006-2010", "2011-2015", "2016-2019", "2021-2025"]
ranges = {"2001-2005": (2001, 2005), "2006-2010": (2006, 2010), "2011-2015": (2011, 2015),
          "2016-2019": (2016, 2019), "2021-2025": (2021, 2025)}

kq_sample = dk_base.dropna(subset=KQ_COV).copy()
print("kq regression sample:", kq_sample.shape)

def wmean(x, w):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    return float(np.sum(w * x) / np.sum(w))

def cell_stats(occ, label, status):
    y0, y1 = ranges[label]
    g = kq_sample[(kq_sample["occupation"] == occ) & kq_sample["YEAR"].between(y0, y1) & (kq_sample["outsourced"] == status)]
    w = g["EARNWT"]
    edu = g["educ_cat"].astype(str)
    out = {
        "real_wage_trimmed": wmean(g["real_wage_trimmed"], w),
        "union": wmean(g["union"], w),
        "parttime": wmean(g["parttime"], w),
        "age": wmean(g["age"], w),
        "female": wmean(g["female"], w),
        "black": wmean(g["black"], w),
        "hispanic": wmean(g["hispanic"], w),
        "edu_lths": wmean(edu.isin(["no_school", "primary", "hs_attend"]).astype(float), w),
        "edu_hs": wmean((edu == "hs_complete").astype(float), w),
        "edu_some": wmean((edu == "some_college").astype(float), w),
        "edu_ba": wmean((edu == "college").astype(float), w),
        "n_unweighted": int(len(g)),
    }
    base = dk_base[(dk_base["occupation"] == occ) & dk_base["YEAR"].between(y0, y1)]
    n_months = base["ym"].nunique()
    out["workers_k"] = round(base.loc[base["outsourced"] == status, "EARNWT"].sum() / n_months / 1000)
    return out, n_months

rows_spec = [
    ("Employment and wages", "Mean hourly wage (2025$)", "real_wage_trimmed", "dollar"),
    ("Employment and wages", "Unionization rate", "union", "pct"),
    ("Employment and wages", "Part-time rate", "parttime", "pct"),
    ("Demographics", "Mean age", "age", "num"),
    ("Demographics", "Female", "female", "pct"),
    ("Demographics", "Black", "black", "pct"),
    ("Demographics", "Hispanic", "hispanic", "pct"),
    ("Education", "Less than high school", "edu_lths", "pct"),
    ("Education", "High school / GED", "edu_hs", "pct"),
    ("Education", "Some college", "edu_some", "pct"),
    ("Education", "Bachelor's or higher", "edu_ba", "pct"),
    ("Sample", "Unweighted N (regression sample)", "n_unweighted", "int"),
    ("Sample", "Workers represented (thousands)", "workers_k", "int"),
]

def fmt(v, kind):
    if kind == "dollar": return f"${v:,.2f}"
    if kind == "pct": return f"{100 * v:.1f}%"
    if kind == "num": return f"{v:.1f}"
    return f"{int(v):,}"

def workforce_table(occ, occ_label, table_no, letter):
    stats = {}
    for label in periods5:
        for status in (0, 1):
            stats[(label, status)], _ = cell_stats(occ, label, status)
    disp_rows, raw_rows = [], []
    for group, measure, key, kind in rows_spec:
        drow, rrow = {"group": group, "measure": measure}, {"group": group, "measure": measure}
        for label in periods5:
            for status, tag in [(0, "in"), (1, "con")]:
                col = f"{label}__{tag}"
                v = stats[(label, status)][key]
                drow[col] = fmt(v, kind)
                rrow[col] = v
        disp_rows.append(drow)
        raw_rows.append(rrow)
    disp, raw = pd.DataFrame(disp_rows), pd.DataFrame(raw_rows)
    gt_obj = (
        GT(disp, rowname_col="measure", groupname_col="group")
        .tab_header(title=f"Table {table_no}: Workforce characteristics, {occ_label}, 2001-2025",
                    subtitle="Contractor = outsourced; In-house = directly employed")
        .tab_source_note(source_note=(
            "Data: IPUMS CPS ORG (outputs/py/dk_clean_v2.parquet). Years: 2001-2025, 2020 excluded. "
            "Weights: EARNWT. Statistics use the kq regression sample (non-missing log_wage and every kq covariate); "
            "unweighted N counts those rows. Workers represented is average monthly employment estimated from EARNWT "
            "on all sampled base-sample workers, including those excluded for missing wages or covariates: sum(EARNWT) "
            "divided by the number of distinct survey months (ym) for that occupation in the period, in thousands, "
            "rounded to the nearest thousand. " + SOURCE_FE))
    )
    for label in periods5:
        gt_obj = gt_obj.tab_spanner(label=label, columns=[f"{label}__in", f"{label}__con"])
    labels = {}
    for label in periods5:
        labels[f"{label}__in"] = "In-house"
        labels[f"{label}__con"] = "Contractor"
    gt_obj = style(gt_obj.cols_label(**labels))
    return gt_obj, raw

gt1, raw1 = workforce_table("janitor", "Janitors", 1, "a")
save_outputs(gt1, raw1, "table1_janitors_workforce_v2")
gt2, raw2 = workforce_table("guard", "Guards", 2, "b")
save_outputs(gt2, raw2, "table2_guards_workforce_v2")

In [ ]:
rows3, raw3 = [], []
for occ, occ_label in [("janitor", "Janitors"), ("guard", "Guards")]:
    m = get_row(estimates, spec="dk", model="base", occ=occ, sample="1983-2000", term="outsourced")
    d = get_row(bench, occ=occ_label, row=1, term="outsourced")
    diff = m["estimate"] - d["estimate"]
    se_diff = math.sqrt(m["se"] ** 2 + d["se"] ** 2)
    z = diff / se_diff
    p_diff = p_two_sided(z)
    d_p = p_two_sided(d["estimate"] / d["se"])
    rows3.append({"occupation": occ_label,
                  "mine_coef": coef_cell(m["estimate"], m["se"], m["p"]),
                  "mine_n": f"{int(m['n_obs']):,}",
                  "dk_coef": coef_cell(d["estimate"], d["se"], d_p),
                  "dk_n": f"{int(d['n']):,}",
                  "diff": f"{diff:.3f}",
                  "p_diff": f"{p_diff:.3f}"})
    raw3.append({"occupation": occ_label, "mine_estimate": m["estimate"], "mine_se": m["se"],
                 "mine_p": m["p"], "mine_n": int(m["n_obs"]), "dk_estimate": d["estimate"],
                 "dk_se": d["se"], "dk_n": int(d["n"]), "diff": diff, "se_diff": se_diff,
                 "z": z, "p_diff": p_diff})
disp3, raw3 = pd.DataFrame(rows3), pd.DataFrame(raw3)
gt3 = style(
    GT(disp3, rowname_col="occupation")
    .tab_header(title="Table 3: D&K pooled 1983-2000 wage penalty, replication",
                subtitle="Outsourced coefficient on log wage, dk spec, row 1")
    .tab_spanner(label="This replication", columns=["mine_coef", "mine_n"])
    .tab_spanner(label="D&K (2010), Table 3", columns=["dk_coef", "dk_n"])
    .tab_spanner(label="Difference (mine - D&K)", columns=["diff", "p_diff"])
    .cols_label(mine_coef="Coef (SE)", mine_n="N", dk_coef="Coef (SE)", dk_n="N",
                diff="Diff.", p_diff="p-value")
    .tab_source_note(source_note=(
        "Data: outputs/py/estimates.csv (this replication); data/comparisons/dk_published_penalty.csv "
        "(D&K row 1, the only source of D&K values). Years: 1983-2000. Weights: EARNWT (replication choice; "
        "D&K do not state weighting). FE: state x year. Clustering: survey month (ym). Stars: * p<.10, ** p<.05, "
        "*** p<.01, two-tailed; D&K stars computed here from the published SE. Difference test: "
        "z = (mine - D&K) / sqrt(SE_mine^2 + SE_D&K^2), two-sided normal p. The test treats the two estimates as "
        "independent, which overstates the SE of the difference since the samples overlap."))
)
save_outputs(gt3, raw3, "table3_dk_pooled_replication_v2")

In [ ]:
period_rows = [("2001-03", "2001-2003"), ("04-06", "2004-2006"), ("07-09", "2007-2009"),
               ("10-12", "2010-2012"), ("13-15", "2013-2015"), ("2016-19", "2016-2019"),
               ("2021-25", "2021-2025"), ("Pooled 2001-2025", "2001-2025")]

def penalty_table(occ, occ_label, table_no):
    disp_rows, raw_rows = [], []
    for label, sample in period_rows:
        dk = get_row(estimates, spec="dk", model="base", occ=occ, sample=sample, term="outsourced")
        kq = get_row(estimates, spec="kq", model="base", occ=occ, sample=sample, term="outsourced")
        assert int(dk["n_obs"]) == int(kq["n_obs"]), (occ, sample)
        disp_rows.append({"period": label,
                          "dk": coef_cell(dk["estimate"], dk["se"], dk["p"]),
                          "kq": coef_cell(kq["estimate"], kq["se"], kq["p"]),
                          "n": f"{int(dk['n_obs']):,}"})
        raw_rows.append({"period": label, "dk_estimate": dk["estimate"], "dk_se": dk["se"], "dk_p": dk["p"],
                         "kq_estimate": kq["estimate"], "kq_se": kq["se"], "kq_p": kq["p"], "n": int(dk["n_obs"])})
    disp, raw = pd.DataFrame(disp_rows), pd.DataFrame(raw_rows)
    gt_obj = style(
        GT(disp, rowname_col="period")
        .tab_header(title=f"Table {table_no}: Outsourcing wage penalty by period, 2001-2025, {occ_label}",
                    subtitle="Outsourced coefficient on log wage, SE in parentheses")
        .tab_spanner(label="Wage penalty", columns=["dk", "kq"])
        .cols_label(dk="dk spec", kq="kq spec (dk + gov_employee)", n="N")
        .tab_source_note(source_note=(
            "Data: outputs/py/estimates.csv (model=base, term=outsourced). Years: each period as labeled; the pooled row "
            "covers 2001-2025 (2020 excluded). Weights: EARNWT. FE: state x year. Clustering: survey month (ym). "
            "Stars: * p<.10, ** p<.05, *** p<.01, two-tailed. dk and kq samples have identical N in every row."))
    )
    return gt_obj, raw

gt4, raw4 = penalty_table("janitor", "Janitors", 4)
save_outputs(gt4, raw4, "table4_janitors_penalty_2001_2025_v2")
gt5, raw5 = penalty_table("guard", "Guards", 5)
save_outputs(gt5, raw5, "table5_guards_penalty_2001_2025_v2")

In [ ]:
term_order = ["outsourced", "union", "parttime",
              "educ_cat[T.no_school]", "educ_cat[T.primary]", "educ_cat[T.hs_attend]",
              "educ_cat[T.hs_complete]", "educ_cat[T.some_college]",
              "female", "black", "hispanic", "age", "age2",
              "msa", "central_city", "cc_unknown", "geo_unidentified", "gov_employee"]
term_label = {
    "outsourced": "Outsourced", "union": "Union", "parttime": "Part-time",
    "educ_cat[T.no_school]": "Education: no schooling (ref.: college)",
    "educ_cat[T.primary]": "Education: primary (ref.: college)",
    "educ_cat[T.hs_attend]": "Education: some high school (ref.: college)",
    "educ_cat[T.hs_complete]": "Education: high school completion (ref.: college)",
    "educ_cat[T.some_college]": "Education: some college (ref.: college)",
    "female": "Female", "black": "Black", "hispanic": "Hispanic", "age": "Age", "age2": "Age squared",
    "msa": "MSA", "central_city": "Central city", "cc_unknown": "MSA, central-city status unknown",
    "geo_unidentified": "Geography unidentified", "gov_employee": "Government employee (kq only)",
}
pooled_cols = [("dk", "1983-2000", "dk_8300"), ("dk", "2001-2025", "dk_0125"),
               ("kq", "1983-2000", "kq_8300"), ("kq", "2001-2025", "kq_0125")]

def appendix_table(occ, occ_label, table_no):
    base = estimates[(estimates["occ"] == occ) & (estimates["model"] == "base")]
    unlabeled = set(base["term"]) - set(term_order)
    assert not unlabeled, f"unlabeled terms: {unlabeled}"
    disp_rows, raw_rows = [], []
    for term in term_order:
        drow, rrow = {"term": term_label[term]}, {"term": term}
        for spec, sample, col in pooled_cols:
            r = base[(base["spec"] == spec) & (base["sample"] == sample) & (base["term"] == term)]
            if len(r) == 0:
                drow[col], rrow[col] = "\u2014", np.nan
            else:
                r = r.iloc[0]
                drow[col] = coef_cell(r["estimate"], r["se"], r["p"])
                rrow[col] = r["estimate"]
        disp_rows.append(drow); raw_rows.append(rrow)
    nrow, rnrow = {"term": "Observations (N)"}, {"term": "Observations (N)"}
    for spec, sample, col in pooled_cols:
        n = int(base[(base["spec"] == spec) & (base["sample"] == sample)]["n_obs"].iloc[0])
        nrow[col], rnrow[col] = f"{n:,}", n
    disp_rows.append(nrow); raw_rows.append(rnrow)
    disp, raw = pd.DataFrame(disp_rows), pd.DataFrame(raw_rows)
    gt_obj = style(
        GT(disp, rowname_col="term")
        .tab_header(title=f"Table A{table_no}: Pooled wage regression coefficients, {occ_label}",
                    subtitle="Dependent variable: log hourly wage. Coefficients with SE in parentheses")
        .tab_spanner(label="dk spec", columns=["dk_8300", "dk_0125"])
        .tab_spanner(label="kq spec (dk + gov_employee)", columns=["kq_8300", "kq_0125"])
        .cols_label(dk_8300="1983-2000", dk_0125="2001-2025", kq_8300="1983-2000", kq_0125="2001-2025")
        .tab_source_note(source_note=(
            "Data: outputs/py/estimates.csv (model=base, pooled samples). Years: 1983-2000 and 2001-2025 (2020 excluded). "
            "Weights: EARNWT. FE: state x year (STATEFIP^YEAR), not shown as rows. Clustering: survey month (ym), CRV1. "
            "Reference education: college. Dashes: term not in model. Stars: * p<.10, ** p<.05, *** p<.01, two-tailed."))
    )
    return gt_obj, raw

gta1, rawa1 = appendix_table("janitor", "Janitors", "1")
save_outputs(gta1, rawa1, "tableA1_janitors_pooled_coefficients_v2")
gta2, rawa2 = appendix_table("guard", "Guards", "2")
save_outputs(gta2, rawa2, "tableA2_guards_pooled_coefficients_v2")

## Regenerate outputs/RESULTS.md

Built from the same CSVs as the tables.

In [ ]:
t3 = pd.read_csv(f"{OUT}/tables/table3_dk_pooled_replication_v2.csv")
t4 = pd.read_csv(f"{OUT}/tables/table4_janitors_penalty_2001_2025_v2.csv")
t5 = pd.read_csv(f"{OUT}/tables/table5_guards_penalty_2001_2025_v2.csv")

def pooled_line(df, occ_name):
    r = df[df["period"] == "Pooled 2001-2025"].iloc[0]
    return f"- {occ_name}: dk {r['dk_estimate']:.3f} (SE {r['dk_se']:.3f}); kq {r['kq_estimate']:.3f} (SE {r['kq_se']:.3f}); N = {int(r['n']):,}"

lines = [
    "# Results: outsourcing wage penalty replication",
    "",
    "Regenerated by `notebooks/03_tables.ipynb` from `outputs/py/`. Do not edit by hand.",
    "",
    "## Tables (`outputs/tables/`, `_v2`: HTML, .tex, CSV)",
    "- Table 1: `table1_janitors_workforce_v2`: janitors, workforce characteristics on the kq regression sample.",
    "- Table 2: `table2_guards_workforce_v2`: guards, same layout.",
    "- Table 3: `table3_dk_pooled_replication_v2`: D&K pooled 1983-2000 penalty, mine vs. published, with difference test.",
    "- Table 4: `table4_janitors_penalty_2001_2025_v2`: janitor penalty by period, dk and kq.",
    "- Table 5: `table5_guards_penalty_2001_2025_v2`: guard penalty by period, dk and kq.",
    "- Table A1: `tableA1_janitors_pooled_coefficients_v2`: janitor pooled coefficients, all terms.",
    "- Table A2: `tableA2_guards_pooled_coefficients_v2`: guard pooled coefficients, all terms.",
    "- PNG: not generated (needs Selenium; Chrome is installed).",
    "",
    "## Headline numbers",
    "Table 3, pooled 1983-2000, dk spec, outsourced coefficient (mine vs. D&K row 1):",
]
for _, r in t3.iterrows():
    lines.append(f"- {r['occupation']}: mine {r['mine_estimate']:.3f} (SE {r['mine_se']:.3f}, N {int(r['mine_n']):,}); "
                 f"D&K {r['dk_estimate']:.3f} (SE {r['dk_se']:.3f}, N {int(r['dk_n']):,}); "
                 f"difference {r['diff']:.3f}, p {r['p_diff']:.3f}")
lines += ["", "Pooled 2001-2025 penalty, dk and kq:", pooled_line(t4, "Janitors"), pooled_line(t5, "Guards"), "",
          "## Caveats",
          "- Replication N is above D&K's published N for both occupations (Table 3). The cause is not established.",
          "- The Table 3 difference test treats the two estimates as independent. The samples overlap, so the test is conservative.",
          "- Tables 1-2 use the kq regression sample, so they can differ slightly from the per-variable summary statistics in 02_estimates section 6.",
          "- Workers represented are estimated from EARNWT on all base-sample workers, not only the regression sample.",
          "- Janitor rows 2-3 (sex split) are not in these tables, so the transposed-label question does not arise here.",
          ""]
with open(f"{OUT}/RESULTS.md", "w") as f:
    f.write("\n".join(lines))
print("wrote outputs/RESULTS.md")